# Car price — Variable selection (all 4 models)
Two layers, decided on **train only** (80/20 split, seed 99). Test is not touched here.

| Layer | Applies to | Method |
|---|---|---|
| 1. Redundancy filter | MLR: every pair ≥ `REDUNDANT_MAX`; trees: exact duplicates only | drop one variable of each redundant pair |
| 2. Model-specific | MLR / DT / RF / XGB | MLR: VIF + backward F-test; trees: permutation importance + backward elimination (KFold 5) |

⚠️ Runs on the current `car_dataset_v5.1.csv` (sha256 `d2c6f2c9…`/LF). Project_note expects `b0540422…` — swap `DATA_PATH` and re-run when that file is available.

In [1]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")   # with all 22 threads, OpenBLAS lstsq runs ~40x slower on this machine
import numpy as np, pandas as pd
from scipy import stats
from sklearn.model_selection import train_test_split
import config as C
from tree_data import load
pd.set_option("display.width", 200)

df = load(drop=False)                     # same cleaning as every model notebook, but year/car_type/brand still present
train, test = train_test_split(df, test_size=0.2, random_state=C.SEED)
NUM = ["engine_capacity", "mileage", "year", "car_age"]
CAT = ["brand", "model", "car_type", "fuel_type", "gear_type", "color"]
print({"train": len(train), "test": len(test)})

removed 287 rows with car_age > 25


{'train': 23853, 'test': 5964}


## 1. Layer 1 — redundancy filter
One measure per pair type, all on a 0–1 scale so one threshold (`REDUNDANT_MAX` = 0.95) fits all:
- **numeric × numeric:** |Pearson r| (Spearman shown as a check for non-linear redundancy)
- **categorical × categorical:** Cramér's V (bias-corrected, Bergsma 2013). V is symmetric, so **which** variable to drop comes from Theil's U: U(a|b) = share of a's uncertainty explained by b. If U(a|b) ≈ 1, b already contains a → drop a (the coarser one).
- **numeric × categorical:** correlation ratio η (√ share of the numeric's variance explained by the category).

In [2]:
def H(x):
    p = x.value_counts(normalize=True); return -(p * np.log(p)).sum()

def theil_u(a, b):
    """U(a|b): share of a's entropy explained by b (1 = b determines a)."""
    return 1 - sum(len(g) / len(a) * H(g) for _, g in a.groupby(b)) / H(a)

def cramers_v(a, b):
    t = pd.crosstab(a, b); n = t.values.sum(); r, k = t.shape
    phi2 = max(0, stats.chi2_contingency(t, correction=False)[0] / n - (k - 1) * (r - 1) / (n - 1))
    rc, kc = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    return np.sqrt(phi2 / min(kc - 1, rc - 1))

def eta(x, g):
    m = x.mean(); return np.sqrt(sum(len(s) * (s.mean() - m) ** 2 for _, s in x.groupby(g)) / ((x - m) ** 2).sum())

rows = []
for i, a in enumerate(NUM):
    for b in NUM[i + 1:]:
        rows.append(dict(a=a, b=b, measure="|r|", value=abs(train[a].corr(train[b])), spearman=abs(train[a].corr(train[b], "spearman"))))
for i, a in enumerate(CAT):
    for b in CAT[i + 1:]:
        rows.append(dict(a=a, b=b, measure="Cramér's V", value=cramers_v(train[a], train[b]), U_a_given_b=theil_u(train[a], train[b]), U_b_given_a=theil_u(train[b], train[a])))
for a in NUM:
    for b in CAT:
        rows.append(dict(a=a, b=b, measure="η", value=eta(train[a], train[b])))
pairs = pd.DataFrame(rows).sort_values("value", ascending=False)
pairs["redundant"] = pairs["value"] >= C.REDUNDANT_MAX
pairs.round(3).head(15)

,a,b,measure,value,spearman,U_a_given_b,U_b_given_a,redundant
5,year,car_age,|r|,1.000,1.00,NaN,NaN,True
6,brand,model,Cramér's V,0.985,NaN,1.000,0.514,True
11,model,car_type,Cramér's V,0.970,NaN,0.350,0.960,True
22,engine_capacity,model,η,0.931,NaN,NaN,NaN,False
12,model,fuel_type,Cramér's V,0.837,NaN,0.155,0.749,False
13,model,gear_type,Cramér's V,0.731,NaN,0.044,0.609,False
16,car_type,gear_type,Cramér's V,0.681,NaN,0.100,0.510,False
23,engine_capacity,car_type,η,0.619,NaN,NaN,NaN,False
34,year,model,η,0.613,NaN,NaN,NaN,False
40,car_age,model,η,0.613,NaN,NaN,NaN,False


### Decision
For each redundant pair, drop the variable the other one already contains:
- numeric pair with |r| = 1 → keep `car_age` (the model notebooks are built on it; `year` = 2026 − car_age carries nothing extra)
- categorical pair → drop `a` if U(a|b) > U(b|a), else `b`

**Which models.** Near-duplicates (≥ `REDUNDANT_MAX` but < 1) are a problem for **MLR only**: collinear columns make its coefficients unstable. Trees are not affected by collinearity, and a coarser copy of a variable can still help them: DT/RF see `model` as integer codes in alphabetical order, so they cannot group a brand's models from `model` alone (CV check: adding `brand` back cuts DT CV MAE by 7%, RF by 14%).
So trees drop only **exact** duplicates (|r| = 1 → `year`) here, and layer 2 decides the rest for them.

In [3]:
drop = {}
for _, p in pairs[pairs.redundant].iterrows():
    if p.measure == "|r|":
        d = "year" if "year" in (p.a, p.b) else p.b
    elif p.measure == "Cramér's V":
        d = p.a if p.U_a_given_b > p.U_b_given_a else p.b
    else:
        raise ValueError(f"η pair over threshold, decide by hand: {p.a} × {p.b}")
    drop.setdefault(d, f"{p.a} × {p.b}: {p.measure} = {p.value:.3f}")
LAYER1 = [c for c in NUM + CAT if c not in drop]                          # MLR
exact = pairs[(pairs.measure == "|r|") & (pairs.value > 1 - 1e-9)]
TREE_START = [c for c in NUM + CAT if c not in {"year" if "year" in (p.a, p.b) else p.b for p in exact.itertuples()}]   # trees
display(pd.Series(drop, name="dropped because"))
print("MLR   -> layer 2:", LAYER1)
print("trees -> layer 2:", TREE_START)

year                 year × car_age: |r| = 1.000
brand          brand × model: Cramér's V = 0.985
car_type    model × car_type: Cramér's V = 0.970
Name: dropped because, dtype: str

MLR   -> layer 2: ['engine_capacity', 'mileage', 'car_age', 'model', 'fuel_type', 'gear_type', 'color']
trees -> layer 2: ['engine_capacity', 'mileage', 'car_age', 'brand', 'model', 'car_type', 'fuel_type', 'gear_type', 'color']


## 2. Layer 2 — model-specific selection
Input: the variables left by layer 1 (`LAYER1` for MLR, `TREE_START` for trees). Everything below uses **train only**, with `KFold(5, shuffle, seed 99)` for every CV number.
Target = `log(price)` for every model; CV MAE is on the **price scale** (as in the model notebooks).
`TOL` = 2%: a removal may cost at most 2% CV MAE, measured against **all layer-1 variables** (not the previous step), so the total loss is capped at 2%.

In [4]:
import time
import statsmodels.api as sm
from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

TOL, VIF_MAX = 0.02, C.VIF_MAX
NUMS = [c for c in NUM if c != "year"]
folds = list(KFold(5, shuffle=True, random_state=C.SEED).split(train))
y = np.log(train[C.TARGET]).values
price = train[C.TARGET].values

### 2a. MLR — VIF, then backward elimination (joint F-test + CV MAE)
**Design.** Numeric columns + one-hot dummies (reference = most frequent level). Fitted with plain OLS on log(price): selection is about **which variables**; the transforms that fix the assumptions (Box-Cox, FGLS, age², brand slopes) are added afterwards in `linear_regression.ipynb`.

**Step 1 — VIF.** A categorical variable has many dummy columns, and their single-column VIFs depend on which level is the reference. So the measure is the **GVIF** (Fox & Monette 1992, *JASA* 87:178–183), one value per variable, invariant to the reference. It is compared with `VIF_MAX` as GVIF^(1/df), i.e. on the same scale as an ordinary VIF.

`engine_capacity` is centred within `model` first (x − model mean of x, from train), as in `linear_regression.ipynb`. Because the model dummies are in X, the model means are already a linear combination of X's columns: the column space and therefore the fit are **identical**, only the collinearity with the model dummies is removed. Both versions are shown.

**Step 2 — backward elimination.** Each round, for every remaining variable: joint F-test (H0: all its coefficients = 0) and CV MAE without it. A variable is dropped only if it is **not significant (p > α)** *and* removing it keeps CV MAE within `TOL` of the all-variables CV MAE — the same "no evidence it helps **and** no measurable cost" rule as the trees in 2b. Among those, drop the one with the largest p; stop when none qualifies.
- Why not "p > α **or** within `TOL`": with ~24k rows almost every p ≈ 0, so that rule is decided by CV alone, and it drops variables the F-test calls highly significant (an earlier run dropped `engine_capacity`, F ≈ 400, because the model dummies already carry most of it: −1.5% CV MAE for 3 variables).

In [5]:
def owner(col):
    return col.split("=")[0]

def mlr_ref(d):
    return dict(levels={v: sorted(d[v].unique()) for v in CAT}, base={v: d[v].mode()[0] for v in CAT},
                cc_mean=d.groupby("model")["engine_capacity"].mean(), cc_all=d["engine_capacity"].mean())

def mlr_X(d, ref, vars_, centre):
    X = d[[v for v in vars_ if v in NUMS]].astype(float)
    if centre and "engine_capacity" in X:
        X["engine_capacity"] = X["engine_capacity"] - d["model"].map(ref["cc_mean"]).fillna(ref["cc_all"]).values
    for v in [v for v in vars_ if v not in NUMS]:
        lv = [l for l in ref["levels"][v] if l != ref["base"][v]]
        X = X.join(pd.DataFrame((d[v].to_numpy(dtype=object)[:, None] == np.array(lv, dtype=object)).astype(float), index=d.index, columns=[f"{v}={l}" for l in lv]))
    return X

def gvif(X, vars_):
    X = X.loc[:, X.std() > 0]; R = np.corrcoef(X.values, rowvar=False); ld = lambda M: np.linalg.slogdet(M)[1]
    g = np.array([owner(c) for c in X.columns]); out = {}
    for v in vars_:
        m = g == v; df_ = int(m.sum())
        out[v] = dict(df=df_, GVIF=np.exp(ld(R[np.ix_(m, m)]) + ld(R[np.ix_(~m, ~m)]) - ld(R)))
    t = pd.DataFrame(out).T; t["GVIF^(1/df)"] = t["GVIF"] ** (1 / t["df"]); return t

ref_all = mlr_ref(train)
vif_tab = pd.concat({"raw": gvif(mlr_X(train, ref_all, LAYER1, False), LAYER1)["GVIF^(1/df)"],
                     "engine_capacity centred in model": gvif(mlr_X(train, ref_all, LAYER1, True), LAYER1)["GVIF^(1/df)"]}, axis=1)
vif_tab["pass (centred)"] = vif_tab.iloc[:, 1] < VIF_MAX
display(vif_tab.round(2))
MLR_VARS = [v for v in LAYER1 if vif_tab.loc[v, "pass (centred)"]]
print("after VIF:", MLR_VARS)

,raw,engine_capacity centred in model,pass (centred)
engine_capacity,8.02,1.07,True
mileage,1.49,1.49,True
car_age,2.26,2.26,True
model,1.02,1.01,True
fuel_type,4.60,4.60,True
gear_type,2.27,2.27,True
color,1.09,1.09,True


after VIF: ['engine_capacity', 'mileage', 'car_age', 'model', 'fuel_type', 'gear_type', 'color']


In [6]:
def mlr_cv(vars_):
    err = []
    for tr, va in folds:
        a, b = train.iloc[tr], train.iloc[va]; ref = mlr_ref(a)
        Xa, Xb = (sm.add_constant(mlr_X(d, ref, vars_, True), has_constant="add") for d in (a, b))
        beta = np.linalg.lstsq(Xa.values, y[tr], rcond=None)[0]
        err.append(np.abs(np.exp(Xb.values @ beta) - price[va]).mean())
    return np.mean(err)

def f_tests(vars_):
    X = sm.add_constant(mlr_X(train, ref_all, vars_, True)); res = sm.OLS(y, X).fit()
    out = {}
    for v in vars_:
        idx = [i for i, c in enumerate(X.columns) if c != "const" and owner(c) == v]
        R = np.zeros((len(idx), X.shape[1])); R[range(len(idx)), idx] = 1
        f = res.f_test(R); out[v] = dict(df=len(idx), F=float(np.squeeze(f.fvalue)), p=float(f.pvalue))
    return pd.DataFrame(out).T

t = time.time()
cur = MLR_VARS.copy(); base = mlr_cv(cur); log = []
while len(cur) > 1:
    ft = f_tests(cur)
    ft["cv_MAE_without"] = [mlr_cv([x for x in cur if x != v]) for v in ft.index]
    ft["loss_vs_all"] = ft["cv_MAE_without"] / base - 1
    ft["droppable"] = (ft["p"] > C.ALPHA) & (ft["loss_vs_all"] <= TOL)
    log.append(ft.assign(round=len(log) + 1))
    if not ft["droppable"].any(): break
    v = ft.loc[ft.droppable, "p"].idxmax()
    cur.remove(v); print(f"round {len(log)}: drop {v}")
MLR_SELECTED = cur
mlr_log = pd.concat(log).reset_index(names="variable").set_index(["round", "variable"])
print(f"{time.time() - t:.0f}s | CV MAE all layer-1 vars: {base:,.0f} | selected: {MLR_SELECTED}")
mlr_log.round(4)

82s | CV MAE all layer-1 vars: 96,418 | selected: ['engine_capacity', 'mileage', 'car_age', 'model', 'fuel_type', 'gear_type', 'color']


df           F    p  cv_MAE_without  loss_vs_all  droppable
round variable                                                                       
1     engine_capacity    1.0    373.9913  0.0      97365.9049       0.0098      False
      mileage            1.0   1092.5175  0.0      99042.2186       0.0272      False
      car_age            1.0  50311.6311  0.0     154525.2200       0.6027      False
      model            462.0    306.7252  0.0     251384.4073       1.6072      False
      fuel_type          2.0    112.1781  0.0      96618.5720       0.0021      False
      gear_type          1.0   1125.4866  0.0      97893.0048       0.0153      False
      color             20.0     12.0323  0.0      96705.1914       0.0030      False

### 2b. DT / RF / XGB — permutation importance + backward elimination
Trees have no p-values, so a variable's worth is measured directly: **permutation importance** = how much CV MAE (price scale) rises when that variable's values are shuffled in the held-out fold (5 folds × 5 shuffles = 25 values per variable).
- Built-in (impurity/gain) importance is not used: it is computed on train and favours variables with many split points (`model` ~500 levels, `mileage`).
- Each round: the least important variable is dropped **only if** its importance is not distinguishable from zero (mean − 2·SD ≤ 0) **and** CV MAE without it stays within `TOL` of the all-variables CV MAE. Otherwise stop.
- Hyperparameters are fixed near the ones tuned in the model notebooks (selection should not depend on tuning; the model notebooks re-tune on the selected variables).
- Encoding as in the model notebooks: integer codes for DT/RF (unseen level → −1), native categorical for XGB.

In [7]:
TREES = {
    "DT": (lambda: DecisionTreeRegressor(max_depth=20, min_samples_leaf=5, random_state=C.SEED), True),
    "RF": (lambda: RandomForestRegressor(n_estimators=200, max_features=0.5, min_samples_leaf=1, n_jobs=-1, random_state=C.SEED), True),
    "XGB": (lambda: XGBRegressor(n_estimators=800, learning_rate=0.05, max_depth=6, min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
                                 enable_categorical=True, tree_method="hist", max_cat_to_onehot=1, random_state=C.SEED, n_jobs=-1), False),
}
rng = np.random.default_rng(C.SEED)

def tree_X(d, fit_d, vars_, codes):
    X = d[vars_].copy()
    for v in [v for v in vars_ if v not in NUMS]:
        lv = sorted(fit_d[v].unique())
        X[v] = pd.Categorical(d[v].where(d[v].isin(lv)), categories=lv)
        if codes: X[v] = X[v].cat.codes
    return X

def tree_cv(make, codes, vars_, perm=True):
    """CV MAE and, if perm, permutation importance (rise in price-scale MAE) of every variable."""
    maes, imp = [], {v: [] for v in vars_}
    for tr, va in folds:
        a, b = train.iloc[tr], train.iloc[va]
        m = make().fit(tree_X(a, a, vars_, codes), y[tr]); Xb = tree_X(b, a, vars_, codes)
        mae = lambda X: np.abs(np.exp(m.predict(X)) - price[va]).mean()
        maes.append(mae(Xb))
        for v in (vars_ if perm else []):
            for _ in range(5):
                Xp = Xb.copy(); Xp[v] = Xp[v].values[rng.permutation(len(Xp))]; imp[v].append(mae(Xp) - maes[-1])
    if not perm: return np.mean(maes), None
    return np.mean(maes), pd.DataFrame({"importance": {v: np.mean(s) for v, s in imp.items()}, "SD": {v: np.std(s) for v, s in imp.items()}})

TREE_SELECTED, tree_logs = {}, {}
for name, (make, codes) in TREES.items():
    t = time.time(); cur = TREE_START.copy(); base, imp = tree_cv(make, codes, cur); log = []
    while len(cur) > 1:
        imp["lower"] = imp["importance"] - 2 * imp["SD"]; v = imp["importance"].idxmin()
        without = tree_cv(make, codes, [x for x in cur if x != v], perm=False)[0]
        ok = bool(imp.loc[v, "lower"] <= 0 and without / base - 1 <= TOL)
        log.append(imp.sort_values("importance", ascending=False)
                   .assign(round=len(log) + 1, cv_MAE_without_weakest=without, loss_vs_all=without / base - 1,
                           weakest=lambda s: s.index == v, dropped=lambda s: (s.index == v) & ok))
        if not ok: break
        cur.remove(v); _, imp = tree_cv(make, codes, cur)
    TREE_SELECTED[name] = cur
    tree_logs[name] = pd.concat(log).reset_index(names="variable").set_index(["round", "variable"])
    print(f"{name}: {time.time() - t:.0f}s | CV MAE all start vars {base:,.0f} | selected {cur}")

DT: 4s | CV MAE all start vars 104,825 | selected ['engine_capacity', 'mileage', 'car_age', 'brand', 'model', 'car_type', 'fuel_type', 'gear_type']


RF: 57s | CV MAE all start vars 83,123 | selected ['engine_capacity', 'mileage', 'car_age', 'brand', 'model', 'car_type', 'fuel_type', 'gear_type', 'color']


XGB: 83s | CV MAE all start vars 75,634 | selected ['engine_capacity', 'mileage', 'car_age', 'brand', 'model', 'car_type', 'fuel_type', 'gear_type', 'color']


In [8]:
for name, log in tree_logs.items():
    print(name); display(log.round(4))

DT


importance          SD        lower  cv_MAE_without_weakest  loss_vs_all  weakest  dropped
round variable                                                                                                    
1     engine_capacity  259514.3733  18825.3539  221863.6655             102865.3449      -0.0187    False    False
      car_age          170385.7389   7540.4975  155304.7438             102865.3449      -0.0187    False    False
      car_type         105606.2322  13934.5426   77737.1471             102865.3449      -0.0187    False    False
      model             97238.0655   6848.9742   83540.1170             102865.3449      -0.0187    False    False
      brand             69354.2800  13240.1949   42873.8902             102865.3449      -0.0187    False    False
      mileage           39542.4252   4074.3310   31393.7632             102865.3449      -0.0187    False    False
      fuel_type         33415.5004   4661.6976   24092.1053             102865.3449      -0.0187    False    False
      gear_type          9638.6936   4234.2102    1170.2731             102865.3449      -0.0187    False    False
      color              1731.3057   1369.6132   -1007.9208             102865.3449      -0.0187     True     True
2     engine_capacity  262230.7102  16534.5603  229161.5896             104605.8142      -0.0021    False    False
      car_age          170891.4375   6530.8126  157829.8122             104605.8142      -0.0021    False    False
      car_type         108076.4880  15663.0876   76750.3127             104605.8142      -0.0021    False    False
      model             97943.4320   7398.9822   83145.4675             104605.8142      -0.0021    False    False
      brand             69637.4764  11421.4105   46794.6555             104605.8142      -0.0021    False    False
      mileage           40556.8181   3216.3064   34124.2052             104605.8142      -0.0021    False    False
      fuel_type         32585.3347   4506.9088   23571.5171             104605.8142      -0.0021    False    False
      gear_type          9465.1465   4640.1615     184.8234             104605.8142      -0.0021     True    False

RF


importance         SD        lower  cv_MAE_without_weakest  loss_vs_all  weakest  dropped
round variable                                                                                                   
1     engine_capacity  194660.4212  5196.2901  184267.8410              84076.1518       0.0115    False    False
      car_age          166588.4925  3314.6436  159959.2053              84076.1518       0.0115    False    False
      car_type          75959.4344  4245.1534   67469.1275              84076.1518       0.0115    False    False
      brand             54405.9102  2801.4522   48803.0057              84076.1518       0.0115    False    False
      model             51023.8733  2357.8879   46308.0976              84076.1518       0.0115    False    False
      mileage           44346.4352  3596.9104   37152.6145              84076.1518       0.0115    False    False
      fuel_type         43359.2692  1862.8246   39633.6200              84076.1518       0.0115    False    False
      gear_type         12345.5955  1426.2637    9493.0682              84076.1518       0.0115    False    False
      color              3032.7812   426.9755    2178.8302              84076.1518       0.0115     True    False

XGB


importance         SD        lower  cv_MAE_without_weakest  loss_vs_all  weakest  dropped
round variable                                                                                                   
1     model            260441.2758  8252.2024  243936.8709              75422.8211      -0.0028    False    False
      car_age          150726.5208  4052.7806  142620.9597              75422.8211      -0.0028    False    False
      mileage           34235.2621  3250.2016   27734.8589              75422.8211      -0.0028    False    False
      engine_capacity   18460.9603  2147.6910   14165.5783              75422.8211      -0.0028    False    False
      brand             13854.6580   934.1072   11986.4436              75422.8211      -0.0028    False    False
      gear_type          5803.8837   402.8888    4998.1060              75422.8211      -0.0028    False    False
      fuel_type          4940.5973   749.3008    3441.9958              75422.8211      -0.0028    False    False
      car_type           3149.4576   500.4245    2148.6087              75422.8211      -0.0028    False    False
      color              2633.3131   876.6822     879.9487              75422.8211      -0.0028     True    False

## 3. Result — variables each model notebook should use

In [9]:
SELECTED = {"MLR": MLR_SELECTED, **TREE_SELECTED}
pd.DataFrame({m: {v: ("✓" if v in s else "–") for v in NUM + CAT} for m, s in SELECTED.items()})

,MLR,DT,RF,XGB
engine_capacity,✓,✓,✓,✓
mileage,✓,✓,✓,✓
year,–,–,–,–
car_age,✓,✓,✓,✓
brand,–,✓,✓,✓
model,✓,✓,✓,✓
car_type,–,✓,✓,✓
fuel_type,✓,✓,✓,✓
gear_type,✓,✓,✓,✓
color,✓,–,✓,✓
